# Korean math datasets: prepare, smoke test, translate, publish
Run on **Colab CPU**. This notebook bundles the tested project modules and needs no repository checkout.
Add `HF_TOKEN` (write scope) and `ANTHROPIC_API_KEY` to Colab Secrets and allow notebook access.
Start with `SMOKE_TEST = True`: all seven datasets are sampled, including the longest retained s1K trace.
Review the downloaded outputs before manually changing `SMOKE_TEST` to `False` and rerunning.
The full run makes paid API requests and publishes seven private datasets under `Seungjun/`.
Source revisions, IDs, translation settings and all quality-check exclusions are recorded in Drive.
Only Spec 1 is implemented. Do not run this notebook simultaneously against the same Drive root.

## 1. Setup and configuration
All run settings are in the next cell; no secrets belong in it.

In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/LG-Korea-AIME'
TRANSLATION_MODEL = 'claude-opus-5-5'
MAX_CONCURRENCY = 16
MAX_OUTPUT_TOKENS = 16000
CHUNK_CHARS = 12000
MAX_RETRIES = 6
SMOKE_TEST = True
SMOKE_TEST_N = 5
HF_USERNAME = 'Seungjun'
HF_PRIVATE = True
SEED = 42
BACKOFF_SECONDS = 1.0
BACKOFF_MAX_SECONDS = 60.0
REQUEST_TIMEOUT_SECONDS = 600.0
LENGTH_RATIO_MIN = 0.3
LENGTH_RATIO_MAX = 2.0
UNSPECIFIED_LICENSE_POLICY = 'block'
DECONTAM_NGRAM_SIZE = 8
DECONTAM_COVERAGE_THRESHOLD = 0.7
DECONTAM_NEAR_MISS_MIN = 0.5

CONFIG = {name: globals()[name] for name in ['PROJECT_ROOT', 'TRANSLATION_MODEL', 'MAX_CONCURRENCY', 'MAX_OUTPUT_TOKENS', 'CHUNK_CHARS', 'MAX_RETRIES', 'SMOKE_TEST', 'SMOKE_TEST_N', 'HF_USERNAME', 'HF_PRIVATE', 'SEED', 'BACKOFF_SECONDS', 'BACKOFF_MAX_SECONDS', 'REQUEST_TIMEOUT_SECONDS', 'LENGTH_RATIO_MIN', 'LENGTH_RATIO_MAX', 'UNSPECIFIED_LICENSE_POLICY', 'DECONTAM_NGRAM_SIZE', 'DECONTAM_COVERAGE_THRESHOLD', 'DECONTAM_NEAR_MISS_MIN']}

# Pinned dataset schema/config/revision settings
DATASETS = {'s1k_1.1': {'repo': 'simplescaling/s1K-1.1', 'revision': '96c411f1fe4c49d20f0e2a1565f61e1a28b0b84d', 'config': 'default', 'split': 'train', 'role': 'train', 'problem_column': 'question', 'answer_column': 'solution', 'id_column': None, 'expected_rows': 1000, 'translate': ['question', 'deepseek_thinking_trajectory', 'deepseek_attempt'], 'upload_name': 's1K-1.1-ko', 'license': 'mit'}, 'dapo_math_17k': {'repo': 'open-r1/DAPO-Math-17k-Processed', 'revision': '31dd309567e3da778038cc87d868b6097a3ccf68', 'config': 'en', 'split': 'train', 'role': 'train', 'problem_column': 'prompt', 'answer_column': 'solution', 'id_column': 'extra_info.index', 'expected_rows': 14116, 'translate': ['problem'], 'upload_name': 'DAPO-Math-17k-ko', 'license': None}, 'aime_2024': {'repo': 'HuggingFaceH4/aime_2024', 'revision': '2fe88a2f1091d5048c0f36abc874fb997b3dd99a', 'config': 'default', 'split': 'train', 'role': 'eval', 'problem_column': 'problem', 'answer_column': 'answer', 'id_column': 'id', 'expected_rows': 30, 'translate': ['problem'], 'upload_name': 'aime_2024-ko', 'license': None}, 'aime_2025': {'repo': 'math-ai/aime25', 'revision': '563bb8404243c5f09de6ec262f2db674fe5bce9b', 'config': 'default', 'split': 'test', 'role': 'eval', 'problem_column': 'problem', 'answer_column': 'answer', 'id_column': 'id', 'expected_rows': 30, 'translate': ['problem'], 'upload_name': 'aime_2025-ko', 'license': 'apache-2.0'}, 'aime_2026': {'repo': 'MathArena/aime_2026', 'revision': 'd2de22f3c656b4f56cf8981212186377d1e23bc3', 'config': 'default', 'split': 'train', 'role': 'eval', 'problem_column': 'problem', 'answer_column': 'answer', 'id_column': 'problem_idx', 'expected_rows': 30, 'translate': ['problem'], 'upload_name': 'aime_2026-ko', 'license': 'cc-by-nc-sa-4.0'}, 'amc23': {'repo': 'math-ai/amc23', 'revision': '80815d37005feb82cd7f8fbc6901d5d3eff43057', 'config': 'default', 'split': 'test', 'role': 'eval', 'problem_column': 'question', 'answer_column': 'answer', 'id_column': 'id', 'expected_rows': 40, 'translate': ['question'], 'upload_name': 'amc23-ko', 'license': None}, 'math_500': {'repo': 'HuggingFaceH4/MATH-500', 'revision': '6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be', 'config': 'default', 'split': 'test', 'role': 'eval', 'problem_column': 'problem', 'answer_column': 'answer', 'id_column': 'unique_id', 'expected_rows': 500, 'translate': ['problem'], 'upload_name': 'MATH-500-ko', 'license': None}}

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "datasets==5.0.1", "huggingface-hub==0.36.2",
                       "anthropic==1.8.0", "PyYAML==6.0.3"])
from google.colab import drive, userdata
drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
if not HF_TOKEN or not ANTHROPIC_API_KEY:
    raise ValueError("Both Colab Secrets are required")

In [ ]:
# @title Install the bundled, tested project code (self-contained notebook)
import base64, io, sys, zipfile
from pathlib import Path
CODE_ROOT = Path("/content/lg-korean-aime")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as bundle:
    bundle.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("Project code ready:", CODE_ROOT)

In [ ]:
from anthropic import AsyncAnthropic
from common.io import write_json
from pipeline.datasets import download_sources
from pipeline.decontamination import prepare_decontamination
from pipeline.translation import Translator, prepare_run, translate_all, check_all
from pipeline.publish import publish, archive_outputs
ROOT = Path(PROJECT_ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
write_json(ROOT / "translation_config.json", CONFIG)
assert TRANSLATION_MODEL and TRANSLATION_MODEL != "TODO"

## 2. Download all datasets
Inspect schemas, preserve source rows, and cache pinned revisions in Drive.

In [ ]:
data, source_manifest = download_sources(DATASETS, ROOT, HF_TOKEN)

## 3. English decontamination
Remove training rows only when one evaluation problem reaches the configured distinct 8-gram coverage threshold. Normalization v2 drops punctuation-only tokens. Versioned caches preserve old outputs. Review per-eval counts, future-contest warnings, borderline removals and near misses below. Evaluation rows stay unchanged.

In [ ]:
from pipeline.decontamination import print_decontamination_report
from pipeline.decontamination_cache import refresh_translation_cache

clean, decontamination_report = prepare_decontamination(
    data, DATASETS, ROOT,
    ngram_size=DECONTAM_NGRAM_SIZE,
    coverage_threshold=DECONTAM_COVERAGE_THRESHOLD,
    near_miss_min=DECONTAM_NEAR_MISS_MIN,
)
print_decontamination_report(decontamination_report)
refresh_translation_cache(data, clean, DATASETS, CONFIG, decontamination_report)

## 4. Translation
Async requests use a semaphore, bounded retries and immediate JSONL persistence.
Truncated chunks are discarded and split smaller at safe boundaries.
Authentication/model errors stop the run. Completed rows are reused after a runtime restart.
`claude-opus-5-5` is the requested model ID; the API must grant your account access to it.

In [ ]:
selected, run_folder, run_manifest = prepare_run(CONFIG, DATASETS, clean)
async with AsyncAnthropic(api_key=ANTHROPIC_API_KEY, max_retries=0,
                          timeout=REQUEST_TIMEOUT_SECONDS) as client:
    translator = Translator(client, CONFIG)
    await translate_all(selected, DATASETS, run_folder, translator)

## 5. Automated checks
Retry each flagged row once; persistent failures are saved and excluded from upload.

In [ ]:
async with AsyncAnthropic(api_key=ANTHROPIC_API_KEY, max_retries=0,
                          timeout=REQUEST_TIMEOUT_SECONDS) as client:
    translator = Translator(client, CONFIG)
    accepted, checks_report = await check_all(
        selected, DATASETS, run_folder, run_manifest, translator)
for name, report in checks_report["datasets"].items():
    print(name, report)

## 6. Assemble, upload, archive and download
In smoke mode this cell **only archives and downloads**; it never uploads or launches a full run.
Review the Korean text, math, reasoning structure, chunk joins and flagged rows in the archive.
After review, manually set `SMOKE_TEST = False` above and rerun the notebook.
Full mode preserves original columns, excludes failed rows, writes source-linked dataset cards,
publishes the seven datasets and saves `eval_suite.json` with immutable uploaded revisions.
Keep that file for baseline and every subsequent stage.

In [ ]:
from google.colab import files
try:
    if SMOKE_TEST:
        print("Smoke test complete. Review the archive before changing SMOKE_TEST to False.")
    else:
        uploads = publish(accepted, DATASETS, CONFIG, decontamination_report,
                          checks_report, run_manifest, HF_TOKEN)
        print(uploads)
finally:
    archive = archive_outputs(ROOT, "smoke_test" if SMOKE_TEST else "translations")
    print("Saved archive:", archive)
    files.download(str(archive))